# FuelEcon_R — Solution

**Short name:** `FuelEcon_R`. Companion to `FuelEcon_R_Practice_Skeleton.ipynb`.

EPA / DOE `vehicles.csv` (this extract: **50,242 rows × 84 columns**, model years **1984–2027**). Book snapshot was ~34,287 × 74, 1984–2014 — follow the *recipes*, not the printed counts.

Reference charts: `fuelecon_r_charts.png`, `fuelecon_r_makes.png`, `fuelecon_r_flowchart.png`.


## 0. Setup


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)

theme_set(theme_minimal(base_size = 12))
options(dplyr.summarise.inform = FALSE)


## 1. Import


In [ ]:
# Book style (base R)
# vehicles <- read.csv("data/vehicles.csv", stringsAsFactors = FALSE)

# Modern equivalent
vehicles <- read_csv("data/vehicles.csv", show_col_types = FALSE)

head(vehicles, 3)
cat("nrow =", nrow(vehicles), " ncol =", ncol(vehicles), "\n")
print(names(vehicles)[1:15])


## 2. Years, fuels, charger type pitfall


In [ ]:
n_years <- length(unique(vehicles$year))
first_year <- min(vehicles$year, na.rm = TRUE)
last_year  <- max(vehicles$year, na.rm = TRUE)
cat("unique years:", n_years, " from", first_year, "to", last_year, "\n")

print(table(vehicles$fuelType1, useNA = "ifany"))

# Super/turbo charger flags — book warned that "T" became logical TRUE
cat("\nclass sCharger:", class(vehicles$sCharger), " unique:", paste(unique(vehicles$sCharger), collapse = " | "), "\n")
cat("class tCharger:", class(vehicles$tCharger), " unique:", paste(unique(vehicles$tCharger), collapse = " | "), "\n")

# Cross-tab turbo presence by decade (updated extract stores "T"/NA, not TRUE)
vehicles <- vehicles %>%
  mutate(
    turbo = ifelse(!is.na(tCharger) & tCharger %in% c("T", "TRUE", TRUE), "Turbo", "No turbo")
  )
print(with(vehicles, table(turbo, year = 10 * (year %/% 10))))


## 3. Transmission → Auto / Manual


In [ ]:
vehicles$trany[vehicles$trany == ""] <- NA
vehicles$trany2 <- ifelse(substr(vehicles$trany, 1, 4) == "Auto", "Auto", "Manual")
vehicles$trany2[is.na(vehicles$trany)] <- NA
print(table(vehicles$trany2, useNA = "ifany"))
# This extract: Auto ~ 36.9k, Manual ~ 13.3k (~2.8 : 1). Book was ~22.5k / 11.8k.


## 4. All-vehicle mean MPG by year

The late spike is real in the *file* but is **not** a gasoline-engine miracle — EVs report very high MPGe in `comb08`.


In [ ]:
mpgByYr <- vehicles %>%
  group_by(year) %>%
  summarise(
    avgMPG  = mean(comb08, na.rm = TRUE),
    avgHghy = mean(highway08, na.rm = TRUE),
    avgCity = mean(city08, na.rm = TRUE),
    n = n()
  )

ggplot(mpgByYr, aes(year, avgMPG)) +
  geom_point(color = "#1f4e79") +
  geom_smooth(se = TRUE, color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(
    x = "Year", y = "Average combined MPG",
    title = "All vehicles — mean combined MPG",
    subtitle = "Includes diesel, hybrid, EV (MPGe). Do not read this as ICE efficiency alone."
  )


## 5. Gasoline-only


In [ ]:
gasCars <- vehicles %>%
  filter(
    fuelType1 %in% c("Regular Gasoline", "Premium Gasoline", "Midgrade Gasoline"),
    is.na(fuelType2) | fuelType2 == "",
    is.na(atvType) | !(atvType %in% c("Hybrid", "Plug-in Hybrid", "EV"))
  )

mpgByYr_Gas <- gasCars %>%
  group_by(year) %>%
  summarise(avgMPG = mean(comb08, na.rm = TRUE), n = n())

ggplot(mpgByYr_Gas, aes(year, avgMPG)) +
  geom_point(color = "#1f4e79") +
  geom_smooth(se = TRUE, color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(
    x = "Year", y = "Average combined MPG",
    title = "Gasoline cars only (no hybrid / dual-fuel / EV)",
    subtitle = "Rise after 2008 is real but far smaller than the all-vehicle line"
  )

# Side-by-side comparison frame
compare <- bind_rows(
  mpgByYr %>% transmute(year, avgMPG, series = "All vehicles"),
  mpgByYr_Gas %>% transmute(year, avgMPG, series = "Gasoline only")
)
ggplot(compare, aes(year, avgMPG, color = series)) +
  geom_line(linewidth = 1) +
  geom_point(size = 1.4) +
  scale_color_manual(values = c("All vehicles" = "#c0392b", "Gasoline only" = "#1f4e79")) +
  labs(x = "Year", y = "Mean combined MPG", color = NULL,
       title = "Why the all-car line misleads after ~2011")


## 6. Displacement vs efficiency


In [ ]:
gasCars$displ <- as.numeric(gasCars$displ)

ggplot(gasCars, aes(displ, comb08)) +
  geom_point(alpha = 0.12, size = 0.7, color = "#2e86ab") +
  geom_smooth(color = "#c0392b") +
  labs(
    x = "Engine displacement (L)", y = "Combined MPG",
    title = "Larger engines → lower MPG (gasoline)"
  )

avgCarSize <- gasCars %>%
  group_by(year) %>%
  summarise(avgDispl = mean(displ, na.rm = TRUE))

ggplot(avgCarSize, aes(year, avgDispl)) +
  geom_point(color = "#6c3483") +
  geom_smooth(color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(x = "Year", y = "Average displacement (L)",
       title = "Average gasoline engine size peaked around 2008")

byYear <- gasCars %>%
  group_by(year) %>%
  summarise(
    `Average MPG` = mean(comb08, na.rm = TRUE),
    `Avg engine displacement` = mean(displ, na.rm = TRUE)
  )

byYear2 <- byYear %>%
  pivot_longer(-year, names_to = "variable", values_to = "value")

ggplot(byYear2, aes(year, value)) +
  geom_point() +
  geom_smooth() +
  facet_wrap(~variable, ncol = 1, scales = "free_y") +
  labs(x = "Year", y = NULL,
       title = "Gasoline fleet: MPG vs engine size on aligned time axes")


## 7. Four-cylinder deep dive


In [ ]:
gasCars4 <- gasCars %>% filter(cylinders == 4)

ggplot(gasCars4, aes(factor(year), comb08)) +
  geom_boxplot(outlier.size = 0.3, fill = "#d6eaf8") +
  facet_wrap(~trany2) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, size = 6)) +
  labs(x = "Year", y = "Combined MPG",
       title = "4-cylinder gasoline MPG by year and transmission")

ggplot(gasCars4 %>% filter(!is.na(trany2)),
       aes(factor(year), fill = factor(trany2))) +
  geom_bar(position = "fill") +
  geom_hline(yintercept = 0.5, linetype = 2) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7)) +
  scale_fill_manual(values = c(Auto = "#1f4e79", Manual = "#f4a261")) +
  labs(x = "Year", y = "Proportion of 4-cyl gasoline models",
       fill = "Transmission",
       title = "Manual share collapsed; automatics dominate late years")


## 8. Makes present every year (book window 1984–2014)


In [ ]:
carsMake <- gasCars4 %>%
  group_by(year) %>%
  summarise(numberOfMakes = n_distinct(make))

ggplot(carsMake, aes(year, numberOfMakes)) +
  geom_point() + geom_line() +
  labs(x = "Year", y = "Unique makes",
       title = "4-cylinder gasoline: number of makes offered")

# Book: dlply + Reduce(intersect)
uniqMakes <- split(gasCars4$make[gasCars4$year <= 2014],
                   gasCars4$year[gasCars4$year <= 2014])
uniqMakes <- lapply(uniqMakes, unique)
commonMakes <- Reduce(intersect, uniqMakes)
print(sort(commonMakes))
cat("n common makes 1984-2014:", length(commonMakes), "\n")
# Expected in this extract: Chevrolet, Chrysler, Dodge, Ford, Honda, Jeep,
# Mazda, Mitsubishi, Nissan, Subaru, Toyota, Volkswagen  (12, same list as the book)

carsCommon <- gasCars4 %>%
  filter(make %in% commonMakes, year <= 2025)

avgMPG_commonMakes <- carsCommon %>%
  group_by(year, make) %>%
  summarise(avgMPG = mean(comb08, na.rm = TRUE))

ggplot(avgMPG_commonMakes, aes(year, avgMPG)) +
  geom_line(color = "#1f4e79") +
  facet_wrap(~make, nrow = 3) +
  geom_vline(xintercept = 2008, linetype = "dotted", color = "grey50") +
  labs(x = "Year", y = "Mean combined MPG",
       title = "12 makes present every year 1984–2014 (4-cyl gasoline)")


## Alternate code

Same year-mean MPG three ways: dplyr (already used), plyr-style, and base `aggregate`.


In [ ]:
# Alternate A — base aggregate (no dplyr)
mpg_base <- aggregate(comb08 ~ year, data = vehicles, FUN = mean)
names(mpg_base)[2] <- "avgMPG"
head(mpg_base)

# Alternate B — tapply
mpg_tapply <- tapply(vehicles$comb08, vehicles$year, mean)
head(mpg_tapply)

# Alternate C — plyr::ddply if the package is installed
# mpg_plyr <- plyr::ddply(vehicles, ~year, plyr::summarise, avgMPG = mean(comb08))

# Alternate D — ggplot computes the mean itself
ggplot(gasCars, aes(year, comb08)) +
  stat_summary(fun = mean, geom = "point") +
  stat_summary(fun = mean, geom = "line") +
  labs(title = "Alternate: stat_summary on raw rows", y = "Mean comb08")

# Alternate E — reshape2::melt instead of tidyr::pivot_longer
# byYear2 <- reshape2::melt(byYear, id = "year")


## More practice — worked


In [ ]:
# 1. City vs highway gasoline means, faceted
city_hwy <- gasCars %>%
  group_by(year) %>%
  summarise(city = mean(city08, na.rm = TRUE),
            highway = mean(highway08, na.rm = TRUE)) %>%
  pivot_longer(-year, names_to = "cycle", values_to = "mpg")

ggplot(city_hwy, aes(year, mpg, color = cycle)) +
  geom_line() +
  labs(title = "Practice 1 — city vs highway (gasoline)", y = "Mean MPG")

# 2. Violin of 4-cyl MPG at three snapshots
snap <- gasCars4 %>% filter(year %in% c(1990, 2010, 2024))
ggplot(snap, aes(factor(year), comb08, fill = factor(year))) +
  geom_violin(trim = FALSE, alpha = 0.7) +
  geom_boxplot(width = 0.12, outlier.shape = NA) +
  labs(title = "Practice 2 — 4-cyl combined MPG distribution", x = "Year", fill = NULL)

# 3. VClass gain 2008 → 2024
gain <- gasCars %>%
  filter(year %in% c(2008, 2024)) %>%
  group_by(VClass, year) %>%
  summarise(m = mean(comb08, na.rm = TRUE), n = n()) %>%
  tidyr::pivot_wider(names_from = year, values_from = c(m, n)) %>%
  mutate(delta = `m_2024` - `m_2008`) %>%
  arrange(desc(delta))
print(head(gain, 8))

# 4. Charger flags
gasCars %>%
  mutate(
    charger = case_when(
      !is.na(sCharger) & sCharger == "S" ~ "Super",
      !is.na(tCharger) & tCharger %in% c("T", "TRUE", TRUE) ~ "Turbo",
      TRUE ~ "Neither"
    )
  ) %>%
  group_by(charger) %>%
  summarise(mean_mpg = mean(comb08, na.rm = TRUE), n = n())

# 5. Book window only
mpg_book_window <- gasCars %>%
  filter(year <= 2014) %>%
  group_by(year) %>%
  summarise(avgMPG = mean(comb08, na.rm = TRUE))
ggplot(mpg_book_window, aes(year, avgMPG)) +
  geom_point() + geom_smooth() +
  labs(title = "Practice 5 — gasoline MPG, 1984–2014 book window")


## Simulation / what-if


In [ ]:
year_max <- 2024
include_hybrids <- FALSE
noise_sd <- 0
sample_frac <- 1.0
set.seed(42)

sim <- vehicles
if (sample_frac < 1) {
  sim <- sim %>% slice_sample(prop = sample_frac)
}
if (noise_sd > 0) {
  sim$comb08 <- sim$comb08 + rnorm(nrow(sim), 0, noise_sd)
}
sim <- sim %>% filter(year <= year_max)

hyb_ok <- if (include_hybrids) {
  function(x) TRUE
} else {
  function(x) is.na(x) | !(x %in% c("Hybrid", "Plug-in Hybrid", "EV"))
}

all_s <- sim %>%
  group_by(year) %>%
  summarise(avgMPG = mean(comb08, na.rm = TRUE), series = "All vehicles")

gas_s <- sim %>%
  filter(
    fuelType1 %in% c("Regular Gasoline", "Premium Gasoline", "Midgrade Gasoline"),
    is.na(fuelType2) | fuelType2 == "",
    hyb_ok(atvType)
  ) %>%
  group_by(year) %>%
  summarise(avgMPG = mean(comb08, na.rm = TRUE), series = "Gasoline only")

both <- bind_rows(all_s, gas_s)

ggplot(both, aes(year, avgMPG, color = series)) +
  geom_line(linewidth = 1) +
  labs(
    title = sprintf("Simulation  year_max=%s  hybrids=%s  noise=%.1f  frac=%.2f",
                    year_max, include_hybrids, noise_sd, sample_frac),
    y = "Mean combined MPG", color = NULL
  )

last_all <- all_s$avgMPG[all_s$year == max(all_s$year)]
last_gas <- gas_s$avgMPG[gas_s$year == max(gas_s$year)]
hit21 <- min(gas_s$year[gas_s$avgMPG > 21], na.rm = TRUE)
cat(sprintf("Latest all-car mean = %.2f | gasoline mean = %.2f | first year gas > 21 MPG = %s\n",
            last_all, last_gas, hit21))


## Audience rewrite — worked example

**1. Analyst.** Combined MPG in the raw EPA file rises from ~19.9 (1984) to the mid-30s by 2024 if every powertrain is pooled. That series mixes MPGe (EVs) with ICE gallons. Restricting to Regular/Premium/Midgrade gasoline, empty `fuelType2`, and non-hybrid `atvType` flattens the curve: ~19.1 MPG in 1984, still ~19.2 in 2008, then ~22.3 by 2014 and ~22.1 in 2024. Displacement is negatively correlated with MPG (r ≈ −0.78). Mean displacement peaked near 2008 (~3.6 L) and eased afterward. Treat `sCharger` / `tCharger` as flags, not numeric; confirm storage class on every extract.

**2. Fleet technician.** If you still buy gasoline four-cylinders, automatics are now ~90% of the 4-cyl catalog (vs ~42% in 1984). Manuals that remain are not the efficiency secret they were in the 1990s; the boxplots move together after 2008. Spec smaller displacement first, then transmission. Turbo prevalence rose; it does not automatically mean worse MPG.

**3. Executive / policy.** Do not brief “the fleet is at 37 MPG” without saying “including EVs.” For a gasoline-majority fleet the honest number is still low-20s combined. CAFE-era downsizing after 2008 shows up in the data; electrification shows up later and dominates the *all-vehicle* line. A procurement target of “match 2014 gasoline mean + 2 MPG” is feasible without waiting for full EV turnover.

**4. Nonspecialist.** Cars on the official list used about 19 miles per gallon in the mid-1980s. Regular gas cars only climbed a few miles per gallon after 2008, when engines got a bit smaller. The big recent jump you hear about comes from counting electric cars, which use a different yardstick. Smaller engine, later model year, and (today) not-only-gasoline are the three simple levers.


## Key numbers (this extract)

| Slice | Result |
|---|---|
| Shape | 50,242 × 84 |
| Years | 1984–2027 (44 unique) |
| fuelType1 | Regular 31,324 · Premium 15,759 · Electricity 1,572 · Diesel 1,310 |
| trany2 | Auto 36,934 · Manual 13,297 |
| Gasoline-only mean MPG | 1984: 19.12 · 2008: 19.19 · 2014: 22.27 · 2024: 22.09 |
| All-vehicle mean MPG | 1984: 19.88 · 2014: 23.55 · 2024: 37.38 |
| Mean displ (gas) | 1984: 3.07 L · 2008: 3.59 L · 2024: 3.12 L |
| 4-cyl Auto share | 1984: 42% · 2024: 91% |
| Common 4-cyl makes 1984–2014 | 12 (same list as the book) |

**Takeaway:** Always split the fleet before celebrating an efficiency miracle. ggplot2 + split–apply–combine makes that split visible.
